In [1]:
import torch, os
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f" GPU {i}:", torch.cuda.get_device_name(i))

CUDA: True | GPUs: 2
 GPU 0: Tesla T4
 GPU 1: Tesla T4


In [2]:
!pip install --quiet --no-deps facenet-pytorch albumentations tqdm scikit-learn matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 30.1 MB/s eta 0:00:00a 0:00:01


In [3]:
import warnings, numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
from torchvision import transforms
from facenet_pytorch import MTCNN
from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix

# Suppress that FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) MTCNN for face alignment
mtcnn = MTCNN(image_size=160, margin=0, device=device)

# 2) Pretrained FaceNet backbone
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
if torch.cuda.device_count()>1:
    resnet = torch.nn.DataParallel(resnet)

# 3) Preprocessing pipeline
tfm = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


  0%|          | 0.00/107M [00:00<?, ?B/s]

In [4]:
root = "/kaggle/input/extyalebcroppedpng/CroppedYalePNG"
files = sorted(f for f in os.listdir(root) if f.lower().endswith((".png",".jpg")))
subjects = sorted({f.split("_")[0] for f in files})
subj2idx = {s:i for i,s in enumerate(subjects)}

image_paths = [os.path.join(root,f) for f in files]
labels      = [subj2idx[f.split("_")[0]] for f in files]

print(f"{len(subjects)} subjects, {len(image_paths)} images")


38 subjects, 2452 images


In [5]:
embs, labs = [], []
for img_path, lbl in tqdm(zip(image_paths, labels), total=len(image_paths), desc="Embedding"):
    img = Image.open(img_path).convert("RGB")
    face = mtcnn(img)
    if face is None:  # skip if no detection
        continue
    with torch.no_grad():
        emb = resnet(face.unsqueeze(0).to(device))[0].cpu().numpy()
    embs.append(emb); labs.append(lbl)

embs = np.stack(embs)    # (N,512)
labs = np.array(labs)    # (N,)
print("Embeddings:", embs.shape, "Labels:", labs.shape)

Embedding: 100%|██████████| 2452/2452 [01:14<00:00, 33.06it/s]

Embeddings: (549, 512) Labels: (549,)


In [9]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

# 1. Count per‐class
counts = Counter(labs)
# 2. Keep only classes with ≥2 samples
valid = {cls for cls,ct in counts.items() if ct >= 2}

# 3. Mask out singletons
mask     = np.isin(labs, list(valid))
embs_filt = embs[mask]
labs_filt = labs[mask]

print(f"Filtered: {embs_filt.shape[0]} embeddings of {len(valid)} classes")

# 4. Stratified split
X_tr, X_te, y_tr, y_te = train_test_split(
    embs_filt, labs_filt,
    test_size=0.2,
    stratify=labs_filt,
    random_state=42
)
print("Train:", X_tr.shape, "Test:", X_te.shape)

Filtered: 548 embeddings of 37 classes
Train: (438, 512) Test: (110, 512)


In [11]:
# Baseline
knn = KNeighborsClassifier(n_neighbors=1, metric='cosine').fit(X_tr, y_tr)
print("KNN baseline:", knn.score(X_te,y_te))

# GridSearch
grid = GridSearchCV(
    KNeighborsClassifier(), 
    {'n_neighbors':[1,3,5],'metric':['cosine','euclidean']},
    cv=5, n_jobs=-1, verbose=0
).fit(X_tr, y_tr)
knn = grid.best_estimator_
print("Best KNN:", grid.best_params_, "→", knn.score(X_te,y_te))

KNN baseline: 0.9636363636363636
Best KNN: {'metric': 'cosine', 'n_neighbors': 1} → 0.9636363636363636


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [12]:
svc = SVC(kernel='rbf',C=1.0,class_weight='balanced').fit(X_tr,y_tr)
print("SVM:", svc.score(X_te,y_te))

logreg = LogisticRegression(max_iter=1000,class_weight='balanced').fit(X_tr,y_tr)
print("LogReg:", logreg.score(X_te,y_te))

SVM: 0.9545454545454546
LogReg: 0.9545454545454546


In [13]:
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

pca = PCA(n_components=100,whiten=True,random_state=42).fit(Xs_tr)
Xp_tr, Xp_te = pca.transform(Xs_tr), pca.transform(Xs_te)

knn_pca = KNeighborsClassifier(n_neighbors=1,metric='cosine').fit(Xp_tr,y_tr)
print("PCA+KNN:", knn_pca.score(Xp_te,y_te))

PCA+KNN: 0.9545454545454546


In [15]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1. Count per‐class in the augmented labels
counts_aug = Counter(labs_aug)

# 2. Keep only classes with ≥2 samples
valid_aug = {cls for cls, ct in counts_aug.items() if ct >= 2}

# 3. Mask out any singletons
mask_aug     = np.isin(labs_aug, list(valid_aug))
embs_aug_f   = embs_aug[mask_aug]
labs_aug_f   = labs_aug[mask_aug]

print(f"After filtering: {embs_aug_f.shape[0]} embeddings of {len(valid_aug)} classes")

# 4. Now you can safely stratify
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    embs_aug_f,
    labs_aug_f,
    test_size=0.2,
    stratify=labs_aug_f,
    random_state=42
)

# 5. Retrain & evaluate
knn2 = KNeighborsClassifier(n_neighbors=1, metric='cosine').fit(X_tr2, y_tr2)
print("Aug+KNN (stratified):", knn2.score(X_te2, y_te2))

After filtering: 567 embeddings of 37 classes
Aug+KNN (stratified): 0.9649122807017544


In [23]:
# Cell 7 (revised): Fine-tune block5 + a larger head for 10 epochs

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np

# 1) Split indices for train/test
idxs = np.arange(len(image_paths))
train_idx, test_idx = train_test_split(
    idxs, test_size=0.2, stratify=labels, random_state=42
)

# 2) Dataset returns aligned face tensor + label
class FaceDataset(Dataset):
    def __init__(self, paths, labs, detector):
        self.paths    = paths
        self.labs     = labs
        self.detector = detector
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        face = self.detector(img)
        if face is None:
            face = torch.zeros(3,160,160)
        return face, self.labs[i]

# 3) DataLoaders (no subprocesses)
train_ds = FaceDataset([image_paths[i] for i in train_idx],
                       [labels[i]      for i in train_idx],
                       detector=mtcnn)
test_ds  = FaceDataset([image_paths[i] for i in test_idx],
                       [labels[i]      for i in test_idx],
                       detector=mtcnn)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)

# 4) Unfreeze block5 of backbone + freeze earlier blocks
for name, param in resnet.named_parameters():
    if "block5" in name or "logits" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# 5) Define a deeper head
n_classes = len(subjects)
head = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, n_classes)
).to(device)

# 6) Optimizer & loss, higher LR
opt     = torch.optim.Adam(list(head.parameters()) + 
                           [p for n,p in resnet.named_parameters() if p.requires_grad],
                           lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# 7) Training + validation loop for 10 epochs
best_acc = 0.0
best_wts = head.state_dict()

for epoch in range(10):
    # — Training
    resnet.train(); head.train()
    running_loss=0
    for faces, ys in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
        faces, ys = faces.to(device), ys.to(device)
        feats = resnet(faces)             # now grads flow into block5+logits
        logits = head(feats)
        loss = loss_fn(logits, ys)
        opt.zero_grad(); loss.backward(); opt.step()
        running_loss += loss.item()*faces.size(0)
    epoch_loss = running_loss/len(train_loader.dataset)

    # — Validation
    resnet.eval(); head.eval()
    correct=0; total=0
    with torch.no_grad():
        for faces, ys in test_loader:
            faces, ys = faces.to(device), ys.to(device)
            logits = head(resnet(faces))
            preds = logits.argmax(dim=1)
            correct += (preds==ys).sum().item()
            total   += ys.size(0)
    epoch_acc = correct/total

    print(f"Epoch {epoch+1}/10 — Loss: {epoch_loss:.4f} — Val Acc: {epoch_acc*100:.2f}%")

    # — Save best
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_wts = {k:v.cpu() for k,v in head.state_dict().items()}

# 8) Load best head weights and save
head.load_state_dict(best_wts)
torch.save({
    'backbone': resnet.state_dict(),
    'head':     head.state_dict(),
}, "facenet_yale_finetuned.pth")

print(f"Best Val Acc: {best_acc*100:.2f}% → saved to facenet_yale_finetuned.pth")

Train Epoch 1: 100%|██████████| 62/62 [00:39<00:00,  1.56it/s]


Epoch 1/10 — Loss: 3.6326 — Val Acc: 5.50%


Train Epoch 2: 100%|██████████| 62/62 [00:39<00:00,  1.55it/s]


Epoch 2/10 — Loss: 3.5984 — Val Acc: 6.31%


Train Epoch 3: 100%|██████████| 62/62 [00:39<00:00,  1.57it/s]


...

Epoch 8/10 — Loss: 3.4508 — Val Acc: 9.37%


Train Epoch 9: 100%|██████████| 62/62 [00:39<00:00,  1.56it/s]


Epoch 9/10 — Loss: 3.4133 — Val Acc: 8.76%


Train Epoch 10: 100%|██████████| 62/62 [00:39<00:00,  1.58it/s]


Epoch 10/10 — Loss: 3.4043 — Val Acc: 10.59%
Best Val Acc: 10.59% → saved to facenet_yale_finetuned.pth


In [24]:
# Cell 8: Final Evaluation of Your Best Fine-Tuned Model

import torch

# 1) Load the checkpoint you just saved
ckpt = torch.load("facenet_yale_finetuned.pth", map_location=device)
resnet.load_state_dict(ckpt['backbone'])
head.load_state_dict(ckpt['head'])

resnet.eval()
head.eval()

# 2) Run on the held-out test set
correct = total = 0
with torch.no_grad():
    for faces, ys in test_loader:
        faces, ys = faces.to(device), ys.to(device)
        feats = resnet(faces)          # (B,512)
        logits = head(feats)           # (B,n_classes)
        preds = logits.argmax(dim=1)   # (B,)
        correct += (preds == ys).sum().item()
        total   += ys.size(0)

print(f"Test Accuracy of Fine-Tuned Model: {correct/total*100:.2f}%")

Test Accuracy of Fine-Tuned Model: 9.78%


In [25]:
# 1. Precompute embeddings on augmented+aligned faces (you already have embs_aug_f, labs_aug_f)

# 2. Stratified split
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    embs_aug_f, labs_aug_f, test_size=0.2, stratify=labs_aug_f, random_state=42
)

# 3. Train 1-NN (cosine)
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1, metric='cosine')
knn.fit(X_tr, y_tr)

# 4. Evaluate
from sklearn.metrics import accuracy_score
y_pred = knn.predict(X_te)
print("Aug+KNN Test Accuracy:", accuracy_score(y_te, y_pred)*100)


Aug+KNN Test Accuracy: 96.49122807017544


In [26]:
import pickle, joblib, os

os.makedirs("checkpoints", exist_ok=True)

# 1) Save embeddings & labels
with open("checkpoints/augmented_embeddings.pkl", "wb") as f:
    pickle.dump({
        "embeddings": embs_aug_f,
        "labels":     labs_aug_f
    }, f)
print("Saved embeddings → checkpoints/augmented_embeddings.pkl")

# 2) Save KNN classifier
joblib.dump(knn, "checkpoints/knn_augmented_cosine.joblib")
print("Saved KNN model → checkpoints/knn_augmented_cosine.joblib")

Saved embeddings → checkpoints/augmented_embeddings.pkl
Saved KNN model → checkpoints/knn_augmented_cosine.joblib


In [27]:
import pickle, joblib

data = pickle.load(open("checkpoints/augmented_embeddings.pkl","rb"))
embs_loaded = data["embeddings"]
labs_loaded = data["labels"]

knn_loaded = joblib.load("checkpoints/knn_augmented_cosine.joblib")

In [28]:
!zip -r checkpoints.zip checkpoints

  adding: checkpoints/ (stored 0%)
  adding: checkpoints/augmented_embeddings.pkl (deflated 8%)
  adding: checkpoints/knn_augmented_cosine.joblib (deflated 8%)


In [ ]:
import pickle, joblib
import torch
from PIL import Image
from facenet_pytorch import MTCNN
from torchvision import transforms
import numpy as np

# 1) Load artifacts
data = pickle.load(open("checkpoints/augmented_embeddings.pkl","rb"))
knn  = joblib.load("checkpoints/knn_augmented_cosine.joblib")

mtcnn = MTCNN(image_size=160, margin=0, device="cuda")
tfm   = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

# 2) Inference helper
def recognize_face(img_path):
    img = Image.open(img_path).convert("RGB")
    face = mtcnn(img)
    if face is None:
        return None
    emb = face.unsqueeze(0).cuda()     # already normalized by mtcnn
    emb = emb.permute(0,3,1,2) if emb.shape[-1]==3 else emb  # ensure CHW
    # get numpy embedding
    with torch.no_grad():
        # if you used face directly with MTCNN→InceptionResnetV1, swap accordingly
        feats = resnet(face.unsqueeze(0).to(device))
    # but here we assume you re-embedded via tfm+resnet:
    # feats = resnet(tfm(img).unsqueeze(0).to(device))
    emb_np = feats[0].cpu().numpy()[np.newaxis,:]  # shape (1,512)
    pred = knn.predict(emb_np)[0]
    return subjects[pred]  # map back to human‐readable ID

# 3) Test on a few held-out examples
for test_img in test_image_list:
    print(test_img, "→", recognize_face(test_img))
